# Stage 2c — TFT (stretch)

Attention-based, parallelizes over sequence positions -- the counter-example to LSTM's limited
GPU speedup. GPU-first: baseline sanity job still on CPU (cheap bug-catch), but HPO runs directly
on GPU since that's the whole point of this model. No separate CPU-vs-GPU comparison run this time
-- that story was already told with DeepAR and LSTM.

Same data (`ts-forecast-demo/deepar-v1/{train,test}`), same results-JSON contract as the LSTM
script, so all three models compare directly.

In [ ]:
%pip install -q "sagemaker<3" boto3 python-dotenv

In [ ]:
import json
import os

import boto3
import sagemaker
from dotenv import load_dotenv
from sagemaker.inputs import TrainingInput
from sagemaker.pytorch import PyTorch
from sagemaker.tuner import ContinuousParameter, HyperparameterTuner, IntegerParameter

load_dotenv()

REGION = boto3.Session().region_name or "us-east-1"
BUCKET = os.environ["BUCKET"]
SAGEMAKER_ROLE = f"arn:aws:iam::{os.environ['ACCOUNT_ID']}:role/ts-forecast-demo-sagemaker-role"

DEEPAR_PREFIX = "ts-forecast-demo/deepar-v1"
TFT_PREFIX = "ts-forecast-demo/tft-v1"
RESULTS_PREFIX = "ts-forecast-demo/results/tft-v1"
CODE_LOCATION = f"s3://{BUCKET}/{TFT_PREFIX}/code"

train_s3 = f"s3://{BUCKET}/{DEEPAR_PREFIX}/train/"
test_s3 = f"s3://{BUCKET}/{DEEPAR_PREFIX}/test/"

CONTEXT_LENGTH = 168
PREDICTION_LENGTH = 168

TAGS = [
    {"Key": "Project", "Value": "ts-forecast-demo"},
    {"Key": "Stage", "Value": "2c-tft-v1"},
    {"Key": "Model", "Value": "tft"},
]

METRIC_DEFINITIONS = [{"Name": "validation:rmse", "Regex": "validation:rmse=([0-9.]+)"}]

session = sagemaker.Session()
s3 = boto3.client("s3")
sm = boto3.client("sagemaker")

## 1. Baseline sanity job (CPU, tiny)

Small hidden_size and 2 epochs only -- this exists purely to catch data/shape/dependency bugs
cheaply before committing to a GPU job. Not meant to produce a good model.

In [ ]:
baseline_hyperparameters = {
    "hidden-size": 16,
    "attention-head-size": 2,
    "hidden-continuous-size": 8,
    "dropout": 0.1,
    "learning-rate": 1e-3,
    "epochs": 2,
    "batch-size": 64,
    "context-length": CONTEXT_LENGTH,
    "prediction-length": PREDICTION_LENGTH,
    "s3-bucket": BUCKET,
    "s3-results-prefix": RESULTS_PREFIX,
}

baseline_estimator = PyTorch(
    entry_point="train.py",
    source_dir="tft_src",
    role=SAGEMAKER_ROLE,
    instance_type="ml.c5.xlarge",
    instance_count=1,
    framework_version="2.1",
    py_version="py310",
    output_path=f"s3://{BUCKET}/{TFT_PREFIX}/model-baseline",
    code_location=CODE_LOCATION,
    hyperparameters=baseline_hyperparameters,
    metric_definitions=METRIC_DEFINITIONS,
    tags=TAGS,
    sagemaker_session=session,
)

baseline_estimator.fit({"train": TrainingInput(train_s3), "test": TrainingInput(test_s3)})

## 2. Hyperparameter tuning (small budget, GPU)

4 jobs, 2 parallel -- smaller than DeepAR/LSTM's 6, since each job is materially more expensive here.
Searching hidden_size, attention_head_size, hidden_continuous_size, learning_rate.

In [ ]:
tuning_estimator = PyTorch(
    entry_point="train.py",
    source_dir="tft_src",
    role=SAGEMAKER_ROLE,
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    framework_version="2.1",
    py_version="py310",
    output_path=f"s3://{BUCKET}/{TFT_PREFIX}/model-hpo",
    code_location=CODE_LOCATION,
    hyperparameters={
        "dropout": 0.1,
        "epochs": 10,
        "batch-size": 64,
        "context-length": CONTEXT_LENGTH,
        "prediction-length": PREDICTION_LENGTH,
        "s3-bucket": BUCKET,
        "s3-results-prefix": RESULTS_PREFIX,
    },
    metric_definitions=METRIC_DEFINITIONS,
    tags=TAGS,
    sagemaker_session=session,
)

hyperparameter_ranges = {
    "hidden-size": IntegerParameter(16, 64),
    "attention-head-size": IntegerParameter(2, 4),
    "hidden-continuous-size": IntegerParameter(8, 32),
    "learning-rate": ContinuousParameter(1e-4, 1e-2),
}

tuner = HyperparameterTuner(
    estimator=tuning_estimator,
    objective_metric_name="validation:rmse",
    objective_type="Minimize",
    hyperparameter_ranges=hyperparameter_ranges,
    metric_definitions=METRIC_DEFINITIONS,
    max_jobs=4,
    max_parallel_jobs=2,
    tags=TAGS,
)

tuner.fit({"train": TrainingInput(train_s3), "test": TrainingInput(test_s3)})

In [ ]:
best_job = tuner.best_training_job()
best_hyperparameters = sagemaker.estimator.Estimator.attach(best_job, sagemaker_session=session).hyperparameters()
print(best_job)
print(best_hyperparameters)

## 3. Cost / instance-time

In [ ]:
job_labels = {
    "baseline (CPU, sanity)": baseline_estimator.latest_training_job.name,
    "best HPO (GPU)": best_job,
}

for name, job_name in job_labels.items():
    desc = sm.describe_training_job(TrainingJobName=job_name)
    print(f"{name:25s} instance={desc['ResourceConfig']['InstanceType']:15s} "
          f"billable_seconds={desc['TrainingTimeInSeconds']}")

## 4. Pull results and persist model card

In [ ]:
def load_job_result(job_name):
    obj = s3.get_object(Bucket=BUCKET, Key=f"{RESULTS_PREFIX}/{job_name}.json")
    return json.loads(obj["Body"].read())

best_result = load_job_result(best_job)
print(f"mean_rmse={best_result['mean_rmse']:.2f} median_rmse={best_result['median_rmse']:.2f}")

model_card = {
    "model": "tft_v1",
    "context_length": CONTEXT_LENGTH,
    "prediction_length": PREDICTION_LENGTH,
    "best_job": best_job,
    "best_hyperparameters": best_hyperparameters,
    "jobs": {
        name: {
            "job_name": job_name,
            "instance_type": sm.describe_training_job(TrainingJobName=job_name)["ResourceConfig"]["InstanceType"],
            "billable_seconds": sm.describe_training_job(TrainingJobName=job_name)["TrainingTimeInSeconds"],
        }
        for name, job_name in job_labels.items()
    },
    "mean_rmse": best_result["mean_rmse"],
    "median_rmse": best_result["median_rmse"],
}

s3.put_object(
    Bucket=BUCKET,
    Key="ts-forecast-demo/results/tft-v1/model_card.json",
    Body=json.dumps(model_card, indent=2).encode(),
)
print("Persisted s3://%s/ts-forecast-demo/results/tft-v1/model_card.json" % BUCKET)

## 5. All three models, side by side

In [ ]:
deepar_card = json.loads(
    s3.get_object(Bucket=BUCKET, Key="ts-forecast-demo/results/deepar-v1/model_card.json")["Body"].read()
)
lstm_card = json.loads(
    s3.get_object(Bucket=BUCKET, Key="ts-forecast-demo/results/lstm-v1/model_card.json")["Body"].read()
)

print(f"{'model':10s} {'mean_rmse':>12s}")
print(f"{'deepar_v1':10s} {deepar_card['evaluation']['mean_rmse']:>12.2f}")
print(f"{'lstm_v1':10s} {lstm_card['jobs']['best HPO (CPU)']['mean_rmse']:>12.2f}")
print(f"{'tft_v1':10s} {model_card['mean_rmse']:>12.2f}")